In [93]:
import requests
import pandas as pd
import time
from tqdm import tqdm
import unicodedata
import re

### Retrieve data

In [94]:
EMAIL = 'okf2003@gmail.com' # Other account mail's244771@dtu.dk'
BASE_URL_AUTHORS = 'https://api.openalex.org/authors'
BASE_URL_WORKS = 'https://api.openalex.org/works'
KEY = '1yqALT6NyxS775KrS65fLl' # Other account ket: 'YU1NMyYLC0SJYghyehfP0i'

In [95]:
df = pd.read_csv('authors.csv') # list of authors from week 1

names = set() # identify unique authors in the author dataset
for entry in df['author'].dropna():
    for name in entry.split(','):
        name = name.strip()
        if name:
            names.add(name)

names = list(names)

print(f'Total unique authors: {len(names)}')

Total unique authors: 1565


### Helper functions, created iteratively after encountering problems

In [96]:
def normalize_name(name):
    # Normalize the name to remove accents and special characters
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    return name

def fallback_name(name):
    # Remove middle initials like "A." or "B.C."
    return re.sub(r'\b[A-Z]\.\s*', '', name).strip()

### Search each name in OpenAlex and retrieve information

In [ ]:
#names = names[:100] # limit to 100 authors for testing; remove this line to process all authors

# Initialize lists to store author information
author_ids = []
display_names = []
works_api_urls = []
h_indexes = []
works_counts = []
country_codes = []


for name in tqdm(names):
    counter = 0
    params = {
        'search': normalize_name(name),
        'mailto': EMAIL,
        'api_key': KEY
    }
    response = requests.get('https://api.openalex.org/authors', params=params).json()
    result = response.get('results', [])

    if not result: # Fallback method: Try to remove the middle initials and search again
        retry_name = fallback_name(normalize_name(name))
        if retry_name != normalize_name(name): # Only retry if the name was actually modified
            params['search'] = retry_name
            response = requests.get('https://api.openalex.org/authors', params=params).json()
            result = response.get('results', [])


    if result:
        author = result[0] # Take the first result as the most relevant match
        author_ids.append(result[0]['id'])
        display_names.append(result[0]['display_name'])
        works_api_urls.append(result[0]['works_api_url'])
        h_indexes.append(author.get('summary_stats', {}).get('h_index')) # Inside summary_stats dict
        works_counts.append(result[0]['works_count'])
        # Get country code from last known institution
        institution = author.get('last_known_institutions', [])
        country_code = institution[0].get('country_code') if institution else None
        country_codes.append(country_code)
    else:
        continue

100%|██████████| 1565/1565 [09:55<00:00,  2.63it/s]


### Store in dataframe

In [116]:
# Store the collected data in a DataFrame
authors_df = pd.DataFrame({
    'author_id': author_ids,
    'display_name': display_names,
    'works_api_url': works_api_urls,
    'h_index': h_indexes,
    'works_count': works_counts,
    'country_code': country_codes
})

# strip url from author_id to get the clean actual id
authors_df['author_id'] = authors_df['author_id'].str.replace('https://openalex.org/', '', regex=False)

In [117]:
# Remove rows with missing author_id and save to csv
authors_df = authors_df.dropna(subset=['author_id'])
authors_df.to_csv('part1_authors_info.csv', index=False)
authors_df

,author_id,display_name,works_api_url,h_index,works_count,country_code
0,A5058810080,Denis Bonnay,https://api.openalex.org/works?filter=author.i...,12.0,72.0,FR
1,A5079881269,Richard Johansson,https://api.openalex.org/works?filter=author.i...,23.0,158.0,None
2,A5021210667,Dominik Brunner,https://api.openalex.org/works?filter=author.i...,66.0,581.0,CH
3,A5033848785,Marlene Lutz,https://api.openalex.org/works?filter=author.i...,3.0,13.0,None
4,A5032446414,Yingdan Lu,https://api.openalex.org/works?filter=author.i...,12.0,47.0,US
...,...,...,...,...,...,...
990,A5100368754,Yuan Zhang,https://api.openalex.org/works?filter=author.i...,71.0,487.0,CN
991,A5005219721,Pier Luigi Sacco,https://api.openalex.org/works?filter=author.i...,33.0,292.0,IT
992,A5058732791,Andreas Bjerre-Nielsen,https://api.openalex.org/works?filter=author.i...,10.0,32.0,DK
993,A5078459283,Catherine Gimbrone,https://api.openalex.org/works?filter=author.i...,11.0,28.0,US


### Reflection questions
Which challenges did you encounter? How did you address them?
We encountered several challenges while identifying the authors in OpenAlex, and here is a few examples:
1. API usage: Without proper queries you quickly run out of API credits. We adressed it by running tests on smaller batches of the dataset, still statistically representative of the dataset. We also created multiple accounts to get more credits
2. Not all the information we were to retrieve could be gathered directly from the api response. We resolved this by e.g. retrieving the authors last known institutions and therefrom the postal code. We had issues finding the h-index, but digging through our output dictionaires, we found it in the sub dictionary called 'summary stats'. 

Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data?

Not all authors in the csv file could be found in OpenAlex by just making an API-call with their name. This was the main problem in this exercise and we solved it by created a set of helper functions. The first called normalize_name strips special characters from the name, e.g. Étienne --> Etienne. The second function called fallback_name removes middle initials from a name - names are not necesarily spelled with middle initial in OpenAlex. By implementing these two function, we reduced the percentage of missing authors in OpenAlex from 55% to 39%.